# 🛒 Superstore Sales Analysis

**Author:** EETAKOTI KARTHIK  
**Tools:** Python · Pandas 
**Dataset:** Sample Superstore (9,994 orders | 2015–2018) from Kaggle.

---

## Project Overview

This notebook performs a simple Data cleaning on a US retail dataset using Python instead of Excel and make easier to load and answer business questions in PostgreSQL.  
The goal is to load and clean the data of Sample Superstore csv file using python and its libraries by analysing:
---

## 1. Import Libraries & Load Data <a id='1'></a>

In [1]:
# Importing Libraries
import pandas as pd

print('Libraries loaded successfully ✅')

Libraries loaded successfully ✅


In [2]:
# Load dataset
df = pd.read_csv('data/superstore.csv', encoding='latin-1')

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

Dataset shape: 9,994 rows × 21 columns


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 2. Data Cleaning & Preparation <a id='2'></a>

In [3]:
# ── Check data types ────────────
print('Data types:')
print(df.dtypes)
print()
print('Missing values:')
print(df.isnull().sum())

Data types:
Row ID             int64
Order ID          object
Order Date        object
Ship Date         object
Ship Mode         object
Customer ID       object
Customer Name     object
Segment           object
Country           object
City              object
State             object
Postal Code        int64
Region            object
Product ID        object
Category          object
Sub-Category      object
Product Name      object
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object

Missing values:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [4]:
# ── Fix date columns ──────────────────────────────────────────────────────────
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')

# ── Engineer new columns ──────────────────────────────────────────────────────
df['Year']          = df['Order Date'].dt.year
df['Month']         = df['Order Date'].dt.month
df['Month_Name']    = df['Order Date'].dt.strftime('%b')
df['Quarter']       = df['Order Date'].dt.quarter
df['YearMonth']     = df['Order Date'].dt.to_period('M')
df['Delivery_Days'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Profit_Margin'] = (df['Profit'] / df['Sales']).round(4)

print('Date columns converted & new features engineered ✅')
print(df[['Order Date', 'Year', 'Month_Name', 'Quarter', 'Delivery_Days', 'Profit_Margin']].head())

Date columns converted & new features engineered ✅
  Order Date  Year Month_Name  Quarter  Delivery_Days  Profit_Margin
0 2016-11-08  2016        Nov        4              3         0.1600
1 2016-11-08  2016        Nov        4              3         0.3000
2 2016-06-12  2016        Jun        2              4         0.4700
3 2015-10-11  2015        Oct        4              7        -0.4000
4 2015-10-11  2015        Oct        4              7         0.1125


In [5]:
# ── Summary statistics ─────────────────────────────────────────────────────────
summary = df[['Sales', 'Quantity', 'Discount', 'Profit', 'Profit_Margin', 'Delivery_Days']].describe().round(2)
print('Numerical summary:')
summary

Numerical summary:


,Sales,Quantity,Discount,Profit,Profit_Margin,Delivery_Days
count,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00
mean,229.86,3.79,0.16,28.66,0.12,3.96
std,623.25,2.23,0.21,234.26,0.47,1.75
min,0.44,1.00,0.00,-6599.98,-2.75,0.00
25%,17.28,2.00,0.00,1.73,0.08,3.00
50%,54.49,3.00,0.20,8.67,0.27,4.00
75%,209.94,5.00,0.20,29.36,0.36,5.00
max,22638.48,14.00,0.80,8399.98,0.50,7.00


## 3. Sales & Profit Overview <a id='3'></a>

In [6]:
# ── High-level KPIs ───────────────────────────────────────────────────────────
kpis = {
    'Total Sales':       f"${df['Sales'].sum():,.0f}",
    'Total Profit':      f"${df['Profit'].sum():,.0f}",
    'Profit Margin':     f"{df['Profit'].sum() / df['Sales'].sum() * 100:.1f}%",
    'Total Orders':      f"{df['Order ID'].nunique():,}",
    'Total Customers':   f"{df['Customer ID'].nunique():,}",
    'Avg Order Value':   f"${df.groupby('Order ID')['Sales'].sum().mean():,.2f}",
    'Avg Delivery Days': f"{df['Delivery_Days'].mean():.1f} days",
}

print('=' * 40)
print('        📊  KEY BUSINESS METRICS')
print('=' * 40)
for k, v in kpis.items():
    print(f'  {k:<22} {v}')
print('=' * 40)

        📊  KEY BUSINESS METRICS
  Total Sales            $2,297,201
  Total Profit           $286,397
  Profit Margin          12.5%
  Total Orders           5,009
  Total Customers        793
  Avg Order Value        $458.61
  Avg Delivery Days      4.0 days


## 4. Connecting to PostgreSQL Server <a id='3'></a>

In [7]:
import pandas as pd

# Loading the dataset
df = pd.read_csv('data/superstore.csv', encoding='latin-1')

# Applying the date conversions
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')
df['Delivery_Days'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Profit_Margin'] = (df['Profit'] / df['Sales']).round(4)

In [ ]:
from sqlalchemy import create_engine
import psycopg2

# Database connection details
DB_USER = "postgres"
DB_PASS = "**************"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "Superstore"

# Connecting via psycopg2 for DDL operations
conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASS,
)
conn.autocommit = True
cur = conn.cursor()

In [ ]:
# ── 1. Createing Normalized Tables ───────────────────────────────────────────────
ddl_schema = """
DROP TABLE IF EXISTS order_items CASCADE;
DROP TABLE IF EXISTS orders CASCADE;
DROP TABLE IF EXISTS products CASCADE;
DROP TABLE IF EXISTS geography CASCADE;
DROP TABLE IF EXISTS customers CASCADE;

CREATE TABLE customers (
    customer_id VARCHAR(20) PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    segment VARCHAR(50) NOT NULL
);

CREATE TABLE geography (
    geo_id SERIAL PRIMARY KEY,
    country VARCHAR(50) NOT NULL,
    city VARCHAR(50) NOT NULL,
    state VARCHAR(50) NOT NULL,
    postal_code VARCHAR(20),
    region VARCHAR(20) NOT NULL,
    CONSTRAINT uq_geo UNIQUE (country, city, state, postal_code, region)
);

CREATE TABLE products (
    product_id VARCHAR(50) PRIMARY KEY,
    category VARCHAR(50) NOT NULL,
    sub_category VARCHAR(50) NOT NULL,
    product_name VARCHAR(255) NOT NULL
);

CREATE TABLE orders (
    order_id VARCHAR(30) PRIMARY KEY,
    order_date DATE NOT NULL,
    ship_date DATE NOT NULL,
    ship_mode VARCHAR(50) NOT NULL,
    customer_id VARCHAR(20) REFERENCES customers(customer_id),
    geo_id INT REFERENCES geography(geo_id)
);

CREATE TABLE order_items (
    row_id INT PRIMARY KEY,
    order_id VARCHAR(30) REFERENCES orders(order_id) ON DELETE CASCADE,
    product_id VARCHAR(50) REFERENCES products(product_id),
    sales NUMERIC(10, 4) NOT NULL,
    quantity INT NOT NULL,
    discount NUMERIC(4, 2) NOT NULL,
    profit NUMERIC(10, 4) NOT NULL
);
"""
cur.execute(ddl_schema)
print("Schema created successfully ✅")


Schema created successfully ✅


In [ ]:
# ── 2. Populating Normalized Tables via Pandas & SQLAlchemy ─────────────────────
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Customers
df_customers = (
    df[["Customer ID", "Customer Name", "Segment"]]
    .drop_duplicates(subset=["Customer ID"])
    .rename(
        columns={
            "Customer ID": "customer_id",
            "Customer Name": "customer_name",
            "Segment": "segment",
        }
    )
)
df_customers.to_sql("customers", engine, if_exists="append", index=False)

# Geography
df_geo = (
    df[["Country", "City", "State", "Postal Code", "Region"]]
    .drop_duplicates()
    .copy()
)
df_geo["Postal Code"] = (
    df_geo["Postal Code"].fillna("00000").astype(str).str.split(".").str[0]
)
df_geo.rename(
    columns={
        "Country": "country",
        "City": "city",
        "State": "state",
        "Postal Code": "postal_code",
        "Region": "region",
    },
    inplace=True,
)
df_geo.to_sql("geography", engine, if_exists="append", index=False)

# Re-reading geography to map surrogate geo_id
geo_lookup = pd.read_sql("SELECT geo_id, country, city, state, postal_code, region FROM geography", engine)

# Products (Handle any duplicate product IDs by keeping the most recent product_name)
df_products = (
    df[["Product ID", "Category", "Sub-Category", "Product Name"]]
    .drop_duplicates(subset=["Product ID"])
    .rename(
        columns={
            "Product ID": "product_id",
            "Category": "category",
            "Sub-Category": "sub_category",
            "Product Name": "product_name",
        }
    )
)
df_products.to_sql("products", engine, if_exists="append", index=False)

# Orders
df_merged = df.copy()
df_merged["Postal Code"] = (
    df_merged["Postal Code"].fillna("00000").astype(str).str.split(".").str[0]
)
df_merged = df_merged.merge(
    geo_lookup,
    left_on=["Country", "City", "State", "Postal Code", "Region"],
    right_on=["country", "city", "state", "postal_code", "region"],
    how="left",
)

df_orders = (
    df_merged[
        [
            "Order ID",
            "Order Date",
            "Ship Date",
            "Ship Mode",
            "Customer ID",
            "geo_id",
        ]
    ]
    .drop_duplicates(subset=["Order ID"])
    .rename(
        columns={
            "Order ID": "order_id",
            "Order Date": "order_date",
            "Ship Date": "ship_date",
            "Ship Mode": "ship_mode",
            "Customer ID": "customer_id",
        }
    )
)
df_orders.to_sql("orders", engine, if_exists="append", index=False)

# Order Items
df_items = df[
    [
        "Row ID",
        "Order ID",
        "Product ID",
        "Sales",
        "Quantity",
        "Discount",
        "Profit",
    ]
].rename(
    columns={
        "Row ID": "row_id",
        "Order ID": "order_id",
        "Product ID": "product_id",
        "Sales": "sales",
        "Quantity": "quantity",
        "Discount": "discount",
        "Profit": "profit",
    }
)
df_items.to_sql("order_items", engine, if_exists="append", index=False)

print("All cleaned data ingested into normalized PostgreSQL tables ✅")

All cleaned data ingested into normalized PostgreSQL tables ✅
